In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder # what is meesage place holder ??
from langchain_core.output_parsers import StrOutputParser
from langchain_core.chat_history import InMemoryChatMessageHistory #in built memory in langchain --
from langchain_core.runnables.history import RunnableWithMessageHistory # wait for 15 mint
from langchain_core.messages import HumanMessage, AIMessage,BaseMessage # by default your LLM try --> input --> human or AI meeage
from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory
from pydantic import BaseModel, Field
from typing import List


In [2]:
load_dotenv()

True

In [5]:

class WindowChatMessageHistory(BaseChatMessageHistory, BaseModel):
    """
    A chat message history that only keeps the last `k` messages.
    
    When the history exceeds k messages, the OLDEST messages
    are silently dropped. This creates a sliding window effect.
    
    Implements BaseChatMessageHistory interface so it works
    seamlessly with RunnableWithMessageHistory.
    """
    messages: List[BaseMessage] = Field(default_factory=list)
    k: int = Field(default=6)  # keep last k messages (= k/2 turns)

    def add_messages(self, messages: List[BaseMessage]) -> None:
        """
        Add new messages, then trim to last k.
        Called automatically by RunnableWithMessageHistory.
        """
        self.messages.extend(messages)
        
        # Trim: keep only the last k messages
        # k=6 means 3 HumanMessages + 3 AIMessages = 3 turns
        if len(self.messages) > self.k:
            dropped = len(self.messages) - self.k
            self.messages = self.messages[-self.k:]
            print(f"  [Window] Dropped {dropped} oldest message(s). Now: {len(self.messages)} messages.")

    def clear(self) -> None:
        """Clear all messages."""
        self.messages = []







In [4]:
message = [3,4]
if len(message)>3:
    message=message[3:]
message


[3, 4]

In [6]:
llm = ChatOpenAI(model="gpt-4o-mini")

In [7]:

WINDOW_K = 4  # keep last 4 messages = 2 full turns (human + AI each)

prompt = ChatPromptTemplate.from_messages([
    ("system", """\
You are a professional email support agent.
Classify: Billing / Technical / General. Priority: High / Medium / Low.
Remember customer details within the conversation."""),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

chain = prompt | llm | StrOutputParser()

In [8]:
window_store = {}

def get_session_history(session_id):
    if session_id not in window_store:
        window_store[session_id] = WindowChatMessageHistory(k=WINDOW_K)
    return window_store[session_id]

In [9]:
# message = ['abc','xyz','abc','I am ok']
# updated_list = []
# k = 2
# if len(message)>k:
#     message = message[k:]

# print(message)

In [10]:
get_session_history("test")

WindowChatMessageHistory(messages=[], k=4)

In [11]:
window_assistant = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history"
)

In [12]:
cfg = {"configurable": {"session_id": "window_john"}}


In [13]:
window_assistant.invoke({"input":"Billing complaint from John — invoice #1042, overcharged $250."},
                        config=cfg)

'Classification: Billing  \nPriority: High\n\nCustomer Details: John, Invoice #1042, overcharged $250.'

In [14]:
turns = [
    "Billing complaint from John — invoice #1042, overcharged $250.",
    "What category is his complaint?",
    "What is the SLA for billing?",         # ← Turn 3 — Turn 1 gets dropped after this
    "Can you remind me who made the original complaint?",  # ← CRITICAL: Turn 1 is now GONE
    "Draft a 2-line apology email for him.",
]


for i, turn in enumerate(turns, 1):
    response = window_assistant.invoke({"input": turn}, config=cfg)
    hist = window_store["window_john"]
    print(response)

Classification: Billing  
Priority: High

Customer Details: John, Invoice #1042, overcharged $250.
  [Window] Dropped 2 oldest message(s). Now: 4 messages.
His complaint falls under the category of Billing.
  [Window] Dropped 2 oldest message(s). Now: 4 messages.
The standard Service Level Agreement (SLA) for billing inquiries typically ranges from 1 to 3 business days for response and resolution. However, please check your organization's specific policies for the most accurate information.
  [Window] Dropped 2 oldest message(s). Now: 4 messages.
I don't have access to specific customer details or previous interactions. If you provide me with the name or email address of the customer, I may be able to assist you further based on that information.
  [Window] Dropped 2 oldest message(s). Now: 4 messages.
Subject: Our Apologies

Dear [Customer's Name], 

I sincerely apologize for any inconvenience caused by the recent issues you've encountered. We appreciate your patience as we work to re

In [7]:
message = ['2','3']

k=2
if len(message)>k:
    print(message[2:])
